In [2]:
!pip install -q generativeai
!pip install -q pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 73.3 MB/s eta 0:00:00


In [3]:
import json
import re
import pdfplumber

from google import genai
from google.genai import types
from google.colab import userdata
from typing import TypedDict, List, Optional

In [4]:
api_key = userdata.get("api_key")

client = genai.Client(api_key=api_key)

In [5]:
class Education(TypedDict):
    degree: str
    institute: str
    year: str
    score: str


class Experience(TypedDict):
    company: str
    role: str
    duration: str
    highlights: List[str]


class Project(TypedDict):
    name: str
    description: str
    tech_stack: List[str]


class Resume(TypedDict):
    name: str
    email: Optional[str]
    phone: Optional[str]
    linkedin_url: Optional[str]
    education: List[Education]
    skills: List[str]
    experience: List[Experience]
    projects: List[Project]
    total_experience_years: float
    summary: str

In [7]:
SYSTEM_PROMPT = """
You are an expert resume parser.

Rules:
1. Extract only information present in the resume.
2. Do not guess missing values.
3. Use null if data is missing.
4. Normalize phone numbers to +91XXXXXXXXXX.
5. Skills must be clean and without duplicates.
6. Summary must be one sentence.
7. Resume may be in any Indian language but output must be in English.
8. Return valid JSON only.
"""

In [6]:
def parse_resume_prompt_only(text):

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=f"""
        {SYSTEM_PROMPT}

        Resume:
        {text}

        Return JSON only.
        """
    )

    return response.text

In [8]:
response_mime_type="application/json"

In [9]:
def parse_resume_mime(text):

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",

        contents=f"""
        {SYSTEM_PROMPT}

        Resume:
        {text}
        """,

        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.2,
            max_output_tokens=2048
        )
    )

    return response.text

In [10]:
def parse_resume_schema(text):

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",

        contents=f"""
        {SYSTEM_PROMPT}

        Resume:
        {text}
        """,

        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Resume,
            temperature=0.2,
            max_output_tokens=2048
        )
    )

    return response.text

In [11]:
def safe_json_parse(text):

    try:
        return json.loads(text)

    except:
        print("Retrying JSON parsing...")

        try:
            return json.loads(text)

        except:
            return {}

In [12]:
def validate_data(data):

    email = data.get("email")

    if email and "@" not in email:
        data["email"] = None

    years = data.get("total_experience_years", 0)

    if years < 0:
        data["total_experience_years"] = 0

    for edu in data.get("education", []):

        year = edu.get("year")

        if year:
            try:
                y = int(year)

                if y < 1990 or y > 2026:
                    edu["year"] = "Invalid"

            except:
                pass

    return data

In [13]:
def parse_resume(text):

    raw = parse_resume_schema(text)

    data = safe_json_parse(raw)

    data = validate_data(data)

    return data

In [15]:
test_resume = """
ANJALI SHARMA
Email: anjali.sharma@gmail.com | Phone: 9876543210
LinkedIn: linkedin.com/in/anjali-sharma
EDUCATION- B.Tech in Computer Science, IIT Delhi, 2024 (CGPA: 8.9)- 12th CBSE, DPS Noida, 2020 (94%)
SKILLS
Python, Django, FastAPI, React.js, MongoDB, PostgreSQL,
Docker, Git, AWS (basics), TensorFlow
EXPERIENCE
Software Engineering Intern, Flipkart  (May 2023 - July 2023)- Built a recommendation microservice using Python + Redis- Reduced API latency by 35%
Open Source Contributor, scikit-learn (2022 - present)- Merged 4 PRs related to model serialization
PROJECTS- AgriBot: WhatsApp chatbot for farmers using Gemini API + Twilio- StudyMate: Note-summarizer Chrome extension (1.2k users)
"""

In [16]:
parsed = parse_resume(test_resume)

print(json.dumps(parsed, indent=2, ensure_ascii=False))

{
  "name": "ANJALI SHARMA",
  "email": "anjali.sharma@gmail.com",
  "phone": "+919876543210",
  "linkedin_url": "linkedin.com/in/anjali-sharma",
  "education": [
    {
      "degree": "B.Tech in Computer Science",
      "institute": "IIT Delhi",
      "year": "2024",
      "score": "8.9"
    },
    {
      "degree": "12th CBSE",
      "institute": "DPS Noida",
      "year": "2020",
      "score": "94%"
    }
  ],
  "skills": [
    "Python",
    "Django",
    "FastAPI",
    "React.js",
    "MongoDB",
    "PostgreSQL",
    "Docker",
    "Git",
    "AWS (basics)",
    "TensorFlow"
  ],
  "experience": [
    {
      "company": "Flipkart",
      "role": "Software Engineering Intern",
      "duration": "May 2023 - July 2023",
      "highlights": [
        "Built a recommendation microservice using Python + Redis",
        "Reduced API latency by 35%"
      ]
    },
    {
      "company": "scikit-learn",
      "role": "Open Source Contributor",
      "duration": "2022 - present",
      "high

In [20]:
print(
    f"""
    Name: {parsed.get('name')}
    Email: {parsed.get('email')}
    Phone: {parsed.get('phone')}
    Skills: {parsed.get('skills')}
    Experience: {parsed.get('total_experience_years')}
    """
)


    Name: ANJALI SHARMA
    Email: anjali.sharma@gmail.com
    Phone: +919876543210
    Skills: ['Python', 'Django', 'FastAPI', 'React.js', 'MongoDB', 'PostgreSQL', 'Docker', 'Git', 'AWS (basics)', 'TensorFlow']
    Experience: 1.5
    


In [17]:
test_resume1='''
hii i am ravi (ravi.k@yahoo.in / 8765-432-100). recently finished
my btech from vit vellore (2023, cgpa around 8). i know java/python/c++
and a bit of ml stuff. did an internship at zoho for 6 months as software
trainee. also worked at a startup called bytemate for 8 months as backend
developer where i built their payment gateway. love football and travelling.
also made a small project called NoteShare - a notes sharing app for
college students built with react and firebase
'''

In [18]:
prased = parse_resume(test_resume1)

print(json.dumps(prased, indent=2, ensure_ascii=False))

{
  "name": "Ravi",
  "email": "ravi.k@yahoo.in",
  "phone": "+918765432100",
  "linkedin_url": null,
  "education": [
    {
      "degree": "B.Tech",
      "institute": "VIT Vellore",
      "year": "2023",
      "score": "8"
    }
  ],
  "skills": [
    "Java",
    "Python",
    "C++",
    "ML",
    "React",
    "Firebase"
  ],
  "experience": [
    {
      "company": "Zoho",
      "role": "Software Trainee",
      "duration": "6 months",
      "highlights": []
    },
    {
      "company": "Bytemate",
      "role": "Backend Developer",
      "duration": "8 months",
      "highlights": [
        "Built their payment gateway"
      ]
    }
  ],
  "projects": [
    {
      "name": "NoteShare",
      "description": "A notes sharing app for college students",
      "tech_stack": [
        "React",
        "Firebase"
      ]
    }
  ],
  "total_experience_years": 1.33,
  "summary": "A recent B.Tech graduate from VIT Vellore with skills in Java, Python, C++, ML, React, and Firebase, seeking

In [19]:
print(
    f"""
    Name: {parsed.get('name')}
    Email: {parsed.get('email')}
    Phone: {parsed.get('phone')}
    Skills: {parsed.get('skills')}
    Experience: {parsed.get('total_experience_years')}
    """
)


    Name: ANJALI SHARMA
    Email: anjali.sharma@gmail.com
    Phone: +919876543210
    Skills: ['Python', 'Django', 'FastAPI', 'React.js', 'MongoDB', 'PostgreSQL', 'Docker', 'Git', 'AWS (basics)', 'TensorFlow']
    Experience: 1.5
    


In [23]:
from google.colab import files

uploaded = files.upload()

Saving Modern Minimalist CV Resume.pdf to Modern Minimalist CV Resume.pdf


In [24]:
pdf_file = list(uploaded.keys())[0]

text = ""

with pdfplumber.open(pdf_file) as pdf:

    for page in pdf.pages:
        text += page.extract_text()

In [25]:
parsed = parse_resume(text)

print(json.dumps(parsed, indent=2))

{
  "name": "Korina Villanueva",
  "email": "hello@reallygreatsite.com",
  "phone": "+911234567890",
  "linkedin_url": null,
  "education": [
    {
      "degree": "Course Studied",
      "institute": "University/College Details",
      "year": "2006 - 2008",
      "score": ""
    },
    {
      "degree": "Course Studied",
      "institute": "University/College Details",
      "year": "2006 - 2008",
      "score": ""
    },
    {
      "degree": "Course Studied",
      "institute": "University/College Details",
      "year": "2006 - 2008",
      "score": ""
    }
  ],
  "skills": [
    "UI/UX",
    "Visual Design",
    "Wireframes",
    "Storyboards",
    "User Flows",
    "Process Flows"
  ],
  "experience": [
    {
      "company": "Company Name",
      "role": "Job position here",
      "duration": "2019 - 2022",
      "highlights": [
        "Lorem ipsum dolor sit amet, consectetur adipiscing elit. Nullam pharetra in lorem at laoreet. Donec hendrerit libero eget est tempor, quis te

In [26]:
print(
    f"""
    Name: {parsed.get('name')}
    Email: {parsed.get('email')}
    Phone: {parsed.get('phone')}
    Skills: {parsed.get('skills')}
    Experience: {parsed.get('total_experience_years')}
    """
)


    Name: Korina Villanueva
    Email: hello@reallygreatsite.com
    Phone: +911234567890
    Skills: ['UI/UX', 'Visual Design', 'Wireframes', 'Storyboards', 'User Flows', 'Process Flows']
    Experience: 9.0
    
